# CSE465 — Notebook 13: Cross-Dataset Generalization (NO Fine-Tuning)

**Model:** trained PVT-MAAG + **TA-DGNN / TADGNN**

**Source training:** modified PVEL-AD only  
**Independent dataset:** PV-Multi-Defect fixed **20% fine-tune / 10% validation / 70% test** split  
**This notebook uses:** **ONLY the untouched 70% test split**  
**Fine-tuning:** **NONE**

This is the official zero-shot baseline for comparison with the later fine-tuned experiment. The model checkpoint is frozen, no optimizer/backward pass is used, and the 20% fine-tune + 10% validation subsets are never loaded for evaluation. All source predictions are collapsed to the single external label `defect` before metrics are computed.

## 1. Match the Notebook 12 Colab environment

In [6]:
import os, sys, time, subprocess
from importlib import metadata

TARGET_TORCH = "2.10.0"
TARGET_TORCHVISION = "0.25.0"
TARGET_TORCHAUDIO = "2.10.0"

gpu_check = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("Use a Colab T4 GPU runtime, then run again.")
print(gpu_check.stdout.strip())

def ver(pkg):
    try: return metadata.version(pkg)
    except metadata.PackageNotFoundError: return "not installed"

if not (ver("torch").startswith(TARGET_TORCH) and ver("torchvision").startswith(TARGET_TORCHVISION)):
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        f"torch=={TARGET_TORCH}", f"torchvision=={TARGET_TORCHVISION}", f"torchaudio=={TARGET_TORCHAUDIO}",
        "--index-url", "https://download.pytorch.org/whl/cu128"
    ], check=True)
    print("PyTorch installed. Runtime will restart once. Re-run all cells after reconnecting.")
    sys.stdout.flush(); time.sleep(2); os.kill(os.getpid(), 9)

print("✅ PyTorch stack ready")

GPU 0: Tesla T4 (UUID: GPU-1ddc39bf-4b4f-337a-e920-32c0e6fcad8b)
✅ PyTorch stack ready


In [7]:
#@title 2) Install project dependencies + exact Mamba CUDA wheel
import os, sys, subprocess
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "setuptools==80.9.0", "wheel", "packaging", "ninja",
    "timm", "torchmetrics", "pycocotools", "einops", "peft==0.20.0",
    "pandas", "matplotlib", "pillow", "pyyaml"
], check=True)

import torch, torchvision
if not torch.__version__.startswith("2.10.0"):
    raise RuntimeError(f"Expected torch 2.10.0, got {torch.__version__}")
if not torchvision.__version__.startswith("0.25.0"):
    raise RuntimeError(f"Expected torchvision 0.25.0, got {torchvision.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Use T4 GPU.")

MAMBA_VERSION = "2.3.2.post1"
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
cuda_major = torch.version.cuda.split(".")[0]
abi = str(torch._C._GLIBCXX_USE_CXX11_ABI).upper()
wheel_name = f"mamba_ssm-{MAMBA_VERSION}+cu{cuda_major}torch{torch_mm}cxx11abi{abi}-{py_tag}-{py_tag}-linux_x86_64.whl"
wheel_url = f"https://github.com/state-spaces/mamba/releases/download/v{MAMBA_VERSION}/{wheel_name}"

try:
    import mamba_ssm
    mamba_ok = str(getattr(mamba_ssm, "__version__", "")).startswith(MAMBA_VERSION)
except Exception:
    mamba_ok = False

if not mamba_ok:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", wheel_url], check=True)

import selective_scan_cuda
from mamba_ssm import Mamba
import peft
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("PEFT:", peft.__version__)
print("✅ Mamba CUDA ready")

Torch: 2.10.0+cu128 CUDA: 12.8
GPU: Tesla T4
PEFT: 0.20.0
✅ Mamba CUDA ready


In [8]:
#@title 3) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Paths and zero-shot evaluation configuration

In [9]:
import gc, json, math, random, shutil, time, warnings, zipfile
from collections import OrderedDict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.retinanet import RetinaNet
from torchvision.ops import FeaturePyramidNetwork, box_iou, nms
from torchvision.ops.feature_pyramid_network import LastLevelP6P7

import timm
from mamba_ssm import Mamba
from peft import AdaLoraConfig, get_peft_model
from torchmetrics.detection.mean_ap import MeanAveragePrecision


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda")

IMAGE_SIZE = 640
BATCH_SIZE = 4
NUM_WORKERS = 2

CONF_THRESHOLD = 0.25
MATCH_IOU_THRESHOLD = 0.50
COLLAPSED_NMS_IOU = 0.50
MAX_DETECTIONS = 100


# Notebook 12 source classes
SOURCE_CLASS_NAMES_FALLBACK = [
    "finger",
    "crack",
    "black core",
    "thick line",
    "horizontal dislocation",
    "short circuit",
    "vertical dislocation",
    "star crack",
    "printing error",
]


# ------------------------------------------------------------
# Locate CSE465 folder
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/Study Materials/CSE465"),
    Path("/content/drive/MyDrive/Study Materials /CSE465"),
]

roots = [p for p in candidate_roots if p.exists()]

if not roots:
    raise FileNotFoundError(
        "CSE465 folder not found.\nChecked:\n"
        + "\n".join(str(p) for p in candidate_roots)
    )

CSE_ROOT = roots[0]


# ------------------------------------------------------------
# Locate trained TA-DGNN checkpoint
# ------------------------------------------------------------

expected_ckpt = (
    CSE_ROOT
    / "12_PVTv2B0_Mamba_ASFF_AdaLoRA_TADGNN_RetinaNet_Aug"
    / "weights"
    / "best.pt"
)

if expected_ckpt.exists():
    TADGNN_BEST_PT = expected_ckpt
else:
    ckpts = [
        p
        for p in CSE_ROOT.rglob("best.pt")
        if "tadgnn" in str(p).lower()
    ]

    if len(ckpts) != 1:
        raise FileNotFoundError(
            f"Could not uniquely find Notebook 12 TADGNN best.pt.\n"
            f"Candidates: {ckpts}"
        )

    TADGNN_BEST_PT = ckpts[0]


# ------------------------------------------------------------
# Locate revised 20/10/70 cross-dataset ZIP
# Exact filename:
# PV_Multi_Defect_cross_dataset.zip
# ------------------------------------------------------------

expected_cross_zip = (
    CSE_ROOT / "PV_Multi_Defect_cross_dataset.zip"
)

if expected_cross_zip.exists():
    CROSS_DATA_ZIP = expected_cross_zip

else:
    # Fallback: search Drive folder for exact filename ignoring case
    candidates = [
        p
        for p in CSE_ROOT.rglob("*.zip")
        if p.name.lower() == "pv_multi_defect_cross_dataset.zip"
    ]

    if len(candidates) != 1:
        raise FileNotFoundError(
            "Could not find PV_Multi_Defect_cross_dataset.zip "
            f"inside:\n{CSE_ROOT}\n"
            f"Found candidates: {candidates}"
        )

    CROSS_DATA_ZIP = candidates[0]


# ------------------------------------------------------------
# Notebook 13 output folders
# ------------------------------------------------------------

EXP_ROOT = (
    CSE_ROOT
    / "13_CrossDataset_NoFineTune_TADGNN_70Test"
)

METRICS_DIR = EXP_ROOT / "metrics"
PLOTS_DIR = EXP_ROOT / "plots"
PRED_DIR = EXP_ROOT / "predictions"

for p in [
    EXP_ROOT,
    METRICS_DIR,
    PLOTS_DIR,
    PRED_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)


print("Checkpoint :", TADGNN_BEST_PT)
print("Dataset ZIP:", CROSS_DATA_ZIP)
print("Protocol   : NO fine-tuning; evaluate ONLY fixed 70% test split")
print("Outputs    :", EXP_ROOT)

Checkpoint : /content/drive/MyDrive/Study Materials /CSE465/12_PVTv2B0_Mamba_ASFF_AdaLoRA_TADGNN_RetinaNet_Aug/weights/best.pt
Dataset ZIP: /content/drive/MyDrive/Study Materials /CSE465/PV_Multi_Defect_cross_dataset.zip
Protocol   : NO fine-tuning; evaluate ONLY fixed 70% test split
Outputs    : /content/drive/MyDrive/Study Materials /CSE465/13_CrossDataset_NoFineTune_TADGNN_70Test


## 5. Load Notebook 12 checkpoint metadata

In [10]:
checkpoint=torch.load(TADGNN_BEST_PT,map_location='cpu',weights_only=False)
if 'model_state' not in checkpoint:
    raise KeyError("Checkpoint has no 'model_state'.")
SOURCE_CLASS_NAMES=list(checkpoint.get('class_names',SOURCE_CLASS_NAMES_FALLBACK))
NUM_CLASSES=len(SOURCE_CLASS_NAMES)
print('Classes:',SOURCE_CLASS_NAMES)
print('Checkpoint epoch:',checkpoint.get('epoch','not stored'))

Classes: ['finger', 'crack', 'black_core', 'thick_line', 'horizontal_dislocation', 'short_circuit', 'vertical_dislocation', 'star_crack', 'printing_error']
Checkpoint epoch: 1


## 6. Exact Notebook 12 model classes

In [11]:
class DeepBidirectionalMamba(nn.Module):
    def __init__(self, channels, d_state=16, d_conv=4, expand=2, gate_init=0.0):
        super().__init__()
        self.channels=channels
        self.norm=nn.LayerNorm(channels)
        self.mamba=Mamba(d_model=channels,d_state=d_state,d_conv=d_conv,expand=expand)
        self.gate=nn.Parameter(torch.tensor(float(gate_init)))
    def forward(self,x):
        b,c,h,w=x.shape
        seq=x.flatten(2).transpose(1,2).contiguous()
        z=self.norm(seq)
        yf=self.mamba(z)
        yr=torch.flip(self.mamba(torch.flip(z,dims=[1])),dims=[1])
        y=(yf+yr)*0.5
        seq=seq+torch.tanh(self.gate)*y
        return seq.transpose(1,2).reshape(b,c,h,w).contiguous()

class ASFFFusionLevel(nn.Module):
    def __init__(self,channels=256,compress_channels=8,num_levels=3,gate_init=0.1):
        super().__init__()
        self.channels=channels; self.num_levels=num_levels
        self.compress=nn.ModuleList([nn.Sequential(nn.Conv2d(channels,compress_channels,1,bias=True),nn.ReLU(inplace=True)) for _ in range(num_levels)])
        self.weight_logits=nn.Conv2d(compress_channels*num_levels,num_levels,1,bias=True)
        nn.init.normal_(self.weight_logits.weight,mean=0.0,std=0.01); nn.init.zeros_(self.weight_logits.bias)
        self.refine=nn.Conv2d(channels,channels,3,padding=1,bias=True)
        self.gate=nn.Parameter(torch.tensor(float(gate_init)))
    def forward(self,aligned_features,target_index):
        compressed=[layer(f) for layer,f in zip(self.compress,aligned_features)]
        weights=torch.softmax(self.weight_logits(torch.cat(compressed,dim=1)),dim=1)
        fused=torch.zeros_like(aligned_features[target_index])
        for i,f in enumerate(aligned_features): fused=fused+weights[:,i:i+1]*f
        fused=self.refine(fused)
        return aligned_features[target_index]+torch.tanh(self.gate)*fused

class AdaptiveSpatialFeatureFusion(nn.Module):
    def __init__(self,channels=256,compress_channels=8,num_levels=3,gate_init=0.1):
        super().__init__()
        self.channels=channels; self.num_levels=num_levels
        self.levels=nn.ModuleList([ASFFFusionLevel(channels,compress_channels,num_levels,gate_init) for _ in range(num_levels)])
    @staticmethod
    def _resize_to(feature,target_hw):
        hw=feature.shape[-2:]
        if tuple(hw)==tuple(target_hw): return feature
        if hw[0]>=target_hw[0] and hw[1]>=target_hw[1]: return F.adaptive_avg_pool2d(feature,target_hw)
        return F.interpolate(feature,size=target_hw,mode='bilinear',align_corners=False)
    def forward(self,features):
        if not isinstance(features,OrderedDict): features=OrderedDict(features)
        keys=list(features.keys())
        if len(keys)<self.num_levels: raise RuntimeError('ASFF requires at least 3 FPN levels.')
        main_keys=keys[:self.num_levels]; main=[features[k] for k in main_keys]
        output=OrderedDict(features)
        for ti,key in enumerate(main_keys):
            target_hw=main[ti].shape[-2:]
            aligned=[self._resize_to(f,target_hw) for f in main]
            output[key]=self.levels[ti](aligned,ti)
        return output

class PVTv2B0MambaASFFFPN(nn.Module):
    def __init__(self,pretrained=False,out_channels=256):
        super().__init__()
        self.body=timm.create_model('pvt_v2_b0',pretrained=pretrained,features_only=True,out_indices=(1,2,3))
        channels=self.body.feature_info.channels()
        if channels[-1]!=256: raise RuntimeError(f'Unexpected PVT deepest channels: {channels[-1]}')
        self.deep_mamba=DeepBidirectionalMamba(channels[-1],d_state=16,d_conv=4,expand=2,gate_init=0.0)
        self.fpn=FeaturePyramidNetwork(channels,out_channels,extra_blocks=LastLevelP6P7(channels[-1],out_channels))
        self.asff=AdaptiveSpatialFeatureFusion(256,8,3,0.1)
        self.out_channels=out_channels
    def forward(self,x):
        features=self.body(x)
        features[-1]=self.deep_mamba(features[-1])
        features=OrderedDict((str(i),f) for i,f in enumerate(features))
        return self.asff(self.fpn(features))

class DynamicTopologyAttentionLayer(nn.Module):
    def __init__(self,dim=256,num_heads=4,k_neighbors=6,geom_hidden=32,mlp_ratio=2.0,dropout=0.0):
        super().__init__()
        if dim%num_heads!=0: raise ValueError('dim must be divisible by num_heads')
        self.dim=dim; self.num_heads=num_heads; self.head_dim=dim//num_heads; self.k_neighbors=k_neighbors
        self.norm1=nn.LayerNorm(dim); self.q_proj=nn.Linear(dim,dim); self.k_proj=nn.Linear(dim,dim); self.v_proj=nn.Linear(dim,dim)
        self.geometry_bias=nn.Sequential(nn.Linear(5,geom_hidden),nn.GELU(),nn.Linear(geom_hidden,num_heads))
        self.out_proj=nn.Linear(dim,dim); self.dropout=nn.Dropout(dropout)
        hidden=int(dim*mlp_ratio); self.norm2=nn.LayerNorm(dim)
        self.ffn=nn.Sequential(nn.Linear(dim,hidden),nn.GELU(),nn.Dropout(dropout),nn.Linear(hidden,dim))
    @staticmethod
    def _relative_geometry(g):
        cx,cy,w,h=g.unbind(-1); eps=1e-6
        dx=cx[None,:]-cx[:,None]; dy=cy[None,:]-cy[:,None]
        lwr=torch.log((w[None,:]+eps)/(w[:,None]+eps)); lhr=torch.log((h[None,:]+eps)/(h[:,None]+eps))
        x1=cx-.5*w; y1=cy-.5*h; x2=cx+.5*w; y2=cy+.5*h
        ix1=torch.maximum(x1[:,None],x1[None,:]); iy1=torch.maximum(y1[:,None],y1[None,:])
        ix2=torch.minimum(x2[:,None],x2[None,:]); iy2=torch.minimum(y2[:,None],y2[None,:])
        inter=(ix2-ix1).clamp_min(0)*(iy2-iy1).clamp_min(0)
        area=(w*h).clamp_min(1e-8); iou=inter/(area[:,None]+area[None,:]-inter+eps)
        return torch.stack([dx,dy,lwr,lhr,iou],dim=-1)
    def forward(self,node_features,geometry):
        n=node_features.shape[0]
        if n<=1: return node_features,None
        z=self.norm1(node_features)
        q=self.q_proj(z).view(n,self.num_heads,self.head_dim).permute(1,0,2)
        k=self.k_proj(z).view(n,self.num_heads,self.head_dim).permute(1,0,2)
        v=self.v_proj(z).view(n,self.num_heads,self.head_dim).permute(1,0,2)
        affinity=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.head_dim)
        affinity=affinity+self.geometry_bias(self._relative_geometry(geometry)).permute(2,0,1)
        diag=torch.eye(n,device=node_features.device,dtype=torch.bool).unsqueeze(0)
        affinity=affinity.masked_fill(diag,float('-inf'))
        kk=min(self.k_neighbors,n-1)
        scores,idx=torch.topk(affinity,k=kk,dim=-1)
        attn=torch.softmax(scores,dim=-1)
        heads=torch.arange(self.num_heads,device=node_features.device)[:,None,None]
        neigh=v[heads,idx]
        agg=(attn.unsqueeze(-1)*neigh).sum(2).permute(1,0,2).reshape(n,self.dim)
        node_features=node_features+self.dropout(self.out_proj(agg))
        node_features=node_features+self.dropout(self.ffn(self.norm2(node_features)))
        return node_features,idx

class TopologyAwareDynamicGNN(nn.Module):
    def __init__(self,feature_dim,num_classes,top_k=128,k_neighbors=6,num_layers=2,num_heads=4,hidden_dim=256,geom_hidden=32,gate_init=0.1,correction_init_std=1e-3,dropout=0.0):
        super().__init__()
        self.feature_dim=feature_dim; self.num_classes=num_classes; self.top_k=top_k
        self.input_projection=nn.Linear(feature_dim,hidden_dim); self.input_norm=nn.LayerNorm(hidden_dim)
        self.geometry_embedding=nn.Sequential(nn.Linear(4,64),nn.GELU(),nn.Linear(64,hidden_dim))
        self.layers=nn.ModuleList([DynamicTopologyAttentionLayer(hidden_dim,num_heads,k_neighbors,geom_hidden,2.0,dropout) for _ in range(num_layers)])
        self.correction_head=nn.Linear(hidden_dim,num_classes)
        nn.init.normal_(self.correction_head.weight,mean=0.0,std=correction_init_std); nn.init.zeros_(self.correction_head.bias)
        self.gate=nn.Parameter(torch.tensor(float(gate_init)))
        self.last_edge_change=None; self.last_selected_score=None; self.last_num_nodes=0
    @staticmethod
    def _normalize_anchor_geometry(boxes,image_size):
        ih,iw=float(image_size[0]),float(image_size[1]); scale=boxes.new_tensor([iw,ih,iw,ih]); b=boxes/scale
        x1,y1,x2,y2=b.unbind(-1); w=(x2-x1).clamp_min(1e-6); h=(y2-y1).clamp_min(1e-6)
        return torch.stack([(x1+x2)*.5,(y1+y2)*.5,w,h],dim=-1)
    @staticmethod
    def _gather_node_features(selected,features,anchors_per_location,batch_index):
        n=selected.numel(); c=features[0].shape[1]; out=features[0].new_zeros((n,c)); offset=0
        for feature in features:
            _,cc,h,w=feature.shape; count=h*w*anchors_per_location
            mask=(selected>=offset)&(selected<offset+count)
            if mask.any():
                pos=torch.where(mask)[0]; local=selected[pos]-offset
                spatial=torch.div(local,anchors_per_location,rounding_mode='floor')
                flat=feature[batch_index].permute(1,2,0).reshape(h*w,cc)
                out=out.index_copy(0,pos,flat[spatial])
            offset+=count
        return out
    @staticmethod
    def _neighbor_change_fraction(prev,cur):
        if prev is None or cur is None: return None
        retained=(prev.unsqueeze(-1)==cur.unsqueeze(-2)).any(-1)
        return 1.0-retained.float().mean()
    def forward(self,cls_logits,features,anchors,image_sizes):
        bs,na,cd=cls_logits.shape
        total_locations=sum(int(f.shape[-2]*f.shape[-1]) for f in features)
        apl=na//total_locations
        sel_scores=torch.sigmoid(cls_logits.detach()).amax(-1)
        n=min(self.top_k,na); selected=torch.topk(sel_scores,k=n,dim=1).indices
        deltas=[]; edge_changes=[]; selected_scores=[]
        for bi in range(bs):
            idx=selected[bi]
            nf=self._gather_node_features(idx,features,apl,bi)
            geom=self._normalize_anchor_geometry(anchors[bi][idx],image_sizes[bi])
            hidden=self.input_norm(self.input_projection(nf))+self.geometry_embedding(geom)
            prev=None
            for layer in self.layers:
                hidden,cur=layer(hidden,geom)
                ch=self._neighbor_change_fraction(prev,cur)
                if ch is not None: edge_changes.append(ch.detach())
                prev=cur
            corr=self.correction_head(hidden)
            delta=torch.zeros_like(cls_logits[bi]).index_copy(0,idx,corr)
            deltas.append(delta); selected_scores.append(sel_scores[bi][idx].mean().detach())
        correction=torch.stack(deltas,0)
        self.last_edge_change=torch.stack(edge_changes).mean() if edge_changes else None
        self.last_selected_score=torch.stack(selected_scores).mean() if selected_scores else None
        self.last_num_nodes=int(n)
        return cls_logits+torch.tanh(self.gate)*correction

class TopologyAwareDynamicRetinaNet(RetinaNet):
    def __init__(self,*args,tadgnn_config=None,**kwargs):
        super().__init__(*args,**kwargs)
        cfg=dict(tadgnn_config or {})
        self.tadgnn=TopologyAwareDynamicGNN(
            feature_dim=self.backbone.out_channels,
            num_classes=self.head.classification_head.num_classes,
            top_k=cfg['top_k'],k_neighbors=cfg['k_neighbors'],num_layers=cfg['num_layers'],num_heads=cfg['num_heads'],
            hidden_dim=cfg['hidden_dim'],geom_hidden=cfg['geometry_hidden_dim'],gate_init=cfg['residual_gate_init'],
            correction_init_std=cfg['correction_head_init_std'],dropout=cfg['dropout'])
    def forward(self,images,targets=None):
        if self.training and targets is None: torch._assert(False,'targets should not be none when in training mode')
        original_image_sizes=[]
        for img in images:
            val=img.shape[-2:]; torch._assert(len(val)==2,'Expected H,W'); original_image_sizes.append((val[0],val[1]))
        images,targets=self.transform(images,targets)
        features=self.backbone(images.tensors)
        if isinstance(features,torch.Tensor): features=OrderedDict([('0',features)])
        feature_list=list(features.values())
        head_outputs=self.head(feature_list); anchors=self.anchor_generator(images,feature_list)
        head_outputs=dict(head_outputs)
        head_outputs['cls_logits']=self.tadgnn(head_outputs['cls_logits'],feature_list,anchors,images.image_sizes)
        losses={}; detections=[]
        if self.training:
            losses=self.compute_loss(targets,head_outputs,anchors)
        else:
            locs=[f.size(2)*f.size(3) for f in feature_list]
            total_locations=sum(locs); total_anchors=head_outputs['cls_logits'].size(1); apl=total_anchors//total_locations
            num_anchors=[x*apl for x in locs]
            split_head={k:list(v.split(num_anchors,dim=1)) for k,v in head_outputs.items()}
            split_anchors=[list(a.split(num_anchors)) for a in anchors]
            detections=self.postprocess_detections(split_head,split_anchors,images.image_sizes)
            detections=self.transform.postprocess(detections,images.image_sizes,original_image_sizes)
        return self.eager_outputs(losses,detections)

## 7. Reconstruct AdaLoRA structure, load `best.pt` strictly, and freeze all parameters

In [12]:
TADGNN_CONFIG={
    'top_k':128,'k_neighbors':6,'num_layers':2,'num_heads':4,'hidden_dim':256,
    'geometry_hidden_dim':32,'residual_gate_init':0.1,'correction_head_init_std':1e-3,'dropout':0.0
}

backbone=PVTv2B0MambaASFFFPN(pretrained=False,out_channels=256)
model=TopologyAwareDynamicRetinaNet(
    backbone=backbone,num_classes=NUM_CLASSES,min_size=IMAGE_SIZE,max_size=IMAGE_SIZE,
    image_mean=[0.485,0.456,0.406],image_std=[0.229,0.224,0.225],
    score_thresh=0.001,detections_per_img=100,tadgnn_config=TADGNN_CONFIG)

# Exact PEFT structure from Notebook 12 / Notebook 09 source detector.
ORIGINAL_TRAIN_IMAGES=4418; ORIGINAL_BATCH_SIZE=8; ORIGINAL_ACCUM_STEPS=2; ORIGINAL_MAX_EPOCHS=25
updates_per_epoch=math.ceil(math.ceil(ORIGINAL_TRAIN_IMAGES/ORIGINAL_BATCH_SIZE)/ORIGINAL_ACCUM_STEPS)
total_steps=ORIGINAL_MAX_EPOCHS*updates_per_epoch
adalora=AdaLoraConfig(
    init_r=12,target_r=8,lora_alpha=32,lora_dropout=0.0,
    target_modules=['attn.q','attn.kv','attn.proj','mlp.fc1','mlp.fc2'],
    tinit=2*updates_per_epoch,tfinal=5*updates_per_epoch,deltaT=25,
    beta1=0.85,beta2=0.85,orth_reg_weight=0.10,total_step=total_steps,bias='none',task_type=None)
model.backbone.body=get_peft_model(model.backbone.body,adalora)

result=model.load_state_dict(checkpoint['model_state'],strict=True)
for p in model.parameters(): p.requires_grad=False
model=model.to(DEVICE); model.eval()

trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
if trainable!=0 or model.training: raise RuntimeError('Zero-shot safeguard failed.')
print(result)
print('Total params:',f"{sum(p.numel() for p in model.parameters()):,}")
print('Trainable params:',trainable)
print('✅ Frozen TADGNN checkpoint loaded')

<All keys matched successfully>
Total params: 15,069,660
Trainable params: 0
✅ Frozen TADGNN checkpoint loaded


## 8. Extract and validate the fixed 20/10/70 external dataset

This cell **requires** the revised split package and selects only `images/test` + `labels/test`. It refuses to fall back to validation and verifies that the package declares the 20/10/70 split.

In [13]:
LOCAL_ROOT=Path('/content/PV_Multi_Defect_20_10_70')
if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)
LOCAL_ROOT.mkdir(parents=True,exist_ok=True)

with zipfile.ZipFile(CROSS_DATA_ZIP,'r') as z:
    z.extractall(LOCAL_ROOT)

# The ZIP may contain files directly or under one wrapper directory.
def locate_dataset_root(root):
    candidates=[]
    for test_dir in root.rglob('test'):
        if test_dir.is_dir() and test_dir.parent.name.lower()=='images':
            dataset_root=test_dir.parent.parent
            required=[
                dataset_root/'images'/'finetune', dataset_root/'labels'/'finetune',
                dataset_root/'images'/'val',      dataset_root/'labels'/'val',
                dataset_root/'images'/'test',     dataset_root/'labels'/'test',
            ]
            if all(p.exists() for p in required):
                candidates.append(dataset_root)
    if len(candidates)!=1:
        raise FileNotFoundError(
            'Expected exactly one processed dataset containing finetune/val/test image+label folders. '
            f'Found: {candidates}'
        )
    return candidates[0]

DATASET_ROOT=locate_dataset_root(LOCAL_ROOT)

# Verify preprocessing metadata when present.
mapping_path=DATASET_ROOT/'mapping.json'
if mapping_path.exists():
    mapping=json.loads(mapping_path.read_text())
    ratios=mapping.get('split_ratios',{})
    expected={'finetune':0.20,'val':0.10,'test':0.70}
    for key,val in expected.items():
        if key not in ratios or abs(float(ratios[key])-val)>1e-9:
            raise RuntimeError(f'Unexpected split ratio for {key}: {ratios.get(key)}; expected {val}')

# Evaluate ONLY the untouched 70% test partition.
EXTERNAL_SPLIT='test'
IMAGE_DIR=DATASET_ROOT/'images'/'test'
LABEL_DIR=DATASET_ROOT/'labels'/'test'

valid_exts={'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
image_paths=sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in valid_exts)
if not image_paths:
    raise RuntimeError('No images found in the fixed 70% test split.')

# Count all three partitions only to verify the package; no finetune/val images are loaded later.
def count_images(split):
    d=DATASET_ROOT/'images'/split
    return len([p for p in d.iterdir() if p.suffix.lower() in valid_exts])

split_counts={s:count_images(s) for s in ['finetune','val','test']}
total=sum(split_counts.values())
print('Dataset root:',DATASET_ROOT)
print('Split counts:',split_counts,'total =',total)
print('Test fraction:',round(split_counts['test']/total,4) if total else None)
print('EVALUATING ONLY:',IMAGE_DIR)
print('Test images:',len(image_paths))

# Optional split-manifest leakage check.
manifest_path=DATASET_ROOT/'split_manifest.csv'
if manifest_path.exists():
    mf=pd.read_csv(manifest_path)
    if 'source_image' in mf.columns and 'split' in mf.columns:
        sets={s:set(mf.loc[mf['split']==s,'source_image'].astype(str)) for s in ['finetune','val','test']}
        assert sets['finetune'].isdisjoint(sets['val'])
        assert sets['finetune'].isdisjoint(sets['test'])
        assert sets['val'].isdisjoint(sets['test'])
        print('✅ Split leakage check passed from split_manifest.csv')

print('✅ No-fine-tuning protocol: finetune and val partitions are NOT used by this notebook.')

Dataset root: /content/PV_Multi_Defect_20_10_70
Split counts: {'finetune': 221, 'val': 110, 'test': 774} total = 1105
Test fraction: 0.7005
EVALUATING ONLY: /content/PV_Multi_Defect_20_10_70/images/test
Test images: 774
✅ Split leakage check passed from split_manifest.csv
✅ No-fine-tuning protocol: finetune and val partitions are NOT used by this notebook.


## 9. External dataset loader

In [14]:
class ExternalYoloDataset(Dataset):
    def __init__(self,image_dir,label_dir):
        self.image_dir=Path(image_dir); self.label_dir=Path(label_dir)
        exts={'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
        self.images=sorted(p for p in self.image_dir.iterdir() if p.suffix.lower() in exts)
    def __len__(self): return len(self.images)
    def __getitem__(self,index):
        path=self.images[index]; pil=Image.open(path).convert('RGB'); w,h=pil.size
        arr=np.asarray(pil,dtype=np.float32)/255.0
        image=torch.from_numpy(arr.transpose(2,0,1)).contiguous()
        boxes=[]; lp=self.label_dir/f'{path.stem}.txt'
        if lp.exists():
            for line in lp.read_text().splitlines():
                parts=line.strip().split()
                if len(parts)<5: continue
                _,cx,cy,bw,bh=map(float,parts[:5])
                x1=max(0.0,min(w,(cx-bw/2)*w)); y1=max(0.0,min(h,(cy-bh/2)*h))
                x2=max(0.0,min(w,(cx+bw/2)*w)); y2=max(0.0,min(h,(cy+bh/2)*h))
                if x2>x1 and y2>y1: boxes.append([x1,y1,x2,y2])
        boxes=torch.tensor(boxes,dtype=torch.float32).reshape(-1,4)
        target={'boxes':boxes,'labels':torch.zeros((len(boxes),),dtype=torch.int64),'image_id':torch.tensor(index)}
        return image,target,str(path)

def collate_fn(batch):
    im,t,p=zip(*batch); return list(im),list(t),list(p)

dataset=ExternalYoloDataset(IMAGE_DIR,LABEL_DIR)
loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,collate_fn=collate_fn,pin_memory=True)
print('External images:',len(dataset))

External images: 774


## 10. Class-agnostic prediction collapse and matching helpers

In [15]:
def collapse_prediction(det,conf_for_map=0.001):
    boxes=det['boxes'].detach().cpu(); scores=det['scores'].detach().cpu()
    keep=scores>=conf_for_map; boxes=boxes[keep]; scores=scores[keep]
    if len(boxes):
        keep_nms=nms(boxes,scores,COLLAPSED_NMS_IOU)[:MAX_DETECTIONS]
        boxes=boxes[keep_nms]; scores=scores[keep_nms]
    return {'boxes':boxes,'scores':scores,'labels':torch.zeros((len(boxes),),dtype=torch.int64)}

def greedy_counts(pred,target,conf=CONF_THRESHOLD,iou_thr=MATCH_IOU_THRESHOLD):
    pboxes=pred['boxes'][pred['scores']>=conf]
    pscores=pred['scores'][pred['scores']>=conf]
    gboxes=target['boxes'].cpu()
    if len(pboxes)==0: return 0,0,len(gboxes),[]
    order=torch.argsort(pscores,descending=True); pboxes=pboxes[order]
    if len(gboxes)==0: return 0,len(pboxes),0,[]
    ious=box_iou(pboxes,gboxes); used=set(); tp=0; matched=[]
    for i in range(len(pboxes)):
        vals,idx=torch.sort(ious[i],descending=True)
        for v,j in zip(vals.tolist(),idx.tolist()):
            if j not in used and v>=iou_thr:
                used.add(j); tp+=1; matched.append(v); break
    fp=len(pboxes)-tp; fn=len(gboxes)-tp
    return tp,fp,fn,matched

## 11. Zero-shot evaluation on the untouched **70% test split only**

The checkpoint remains frozen. This cell performs inference only and computes class-agnostic external metrics.

In [16]:
metric=MeanAveragePrecision(box_format='xyxy',iou_type='bbox',class_metrics=False)
TP=FP=FN=0; matched_ious=[]; records=[]; all_visual=[]

# Warm-up
with torch.inference_mode():
    sample_img,_,_=dataset[0]
    for _ in range(3): _=model([sample_img.to(DEVICE)])
torch.cuda.synchronize()

start=time.perf_counter(); total_images=0
with torch.inference_mode():
    for batch_idx,(images,targets,paths) in enumerate(loader):
        gpu_images=[x.to(DEVICE,non_blocking=True) for x in images]
        outputs=model(gpu_images)
        collapsed=[]; metric_targets=[]
        for out,t,path in zip(outputs,targets,paths):
            pred=collapse_prediction(out,conf_for_map=0.001)
            target={'boxes':t['boxes'].cpu(),'labels':torch.zeros((len(t['boxes']),),dtype=torch.int64)}
            collapsed.append(pred); metric_targets.append(target)
            tp,fp,fn,mios=greedy_counts(pred,target)
            TP+=tp; FP+=fp; FN+=fn; matched_ious.extend(mios)
            records.append({'image':Path(path).name,'gt_boxes':len(target['boxes']),'pred_boxes_map':len(pred['boxes']),'pred_boxes_conf025':int((pred['scores']>=CONF_THRESHOLD).sum()),'tp':tp,'fp':fp,'fn':fn})
            if len(all_visual)<12: all_visual.append((path,pred,target))
        metric.update(collapsed,metric_targets)
        total_images+=len(images)
        if (batch_idx+1)%20==0: print(f'Processed {total_images}/{len(dataset)}')

torch.cuda.synchronize(); elapsed=time.perf_counter()-start
map_result=metric.compute()

precision=TP/(TP+FP) if TP+FP else 0.0
recall=TP/(TP+FN) if TP+FN else 0.0
f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
mean_iou=float(np.mean(matched_ious)) if matched_ious else float('nan')

results={
    'model':'PVTv2-B0 + BiMamba + FPN + ASFF + AdaLoRA + RetinaNet + TADGNN',
    'protocol':'cross-dataset zero-shot / no fine-tuning / class-agnostic defect localization',
    'external_dataset':'PV-Multi-Defect processed',
    'external_split':EXTERNAL_SPLIT,
    'external_split_protocol':'fixed 20% finetune / 10% val / 70% test; evaluated on test only',
    'images':len(dataset),
    'precision_at_0.25':precision,'recall_at_0.25':recall,'f1_at_0.25':f1,
    'map50':float(map_result['map_50']),'map75':float(map_result['map_75']),'map50_95':float(map_result['map']),
    'matched_mean_iou_at_0.25_iou0.50':mean_iou,
    'tp':TP,'fp':FP,'fn':FN,
    'elapsed_seconds':elapsed,'ms_per_image':1000*elapsed/len(dataset),'fps':len(dataset)/elapsed,
    'confidence_for_prf1':CONF_THRESHOLD,'match_iou':MATCH_IOU_THRESHOLD,'collapsed_nms_iou':COLLAPSED_NMS_IOU,
    'fine_tuning':False,'trainable_parameters':0,
    'finetune_images_used':0,'validation_images_used':0,'test_images_used':len(dataset),
}

pd.DataFrame([results]).to_csv(METRICS_DIR/'cross_dataset_zero_shot_metrics.csv',index=False)
pd.DataFrame(records).to_csv(METRICS_DIR/'per_image_results.csv',index=False)
with open(METRICS_DIR/'cross_dataset_zero_shot_metrics.json','w') as f: json.dump(results,f,indent=2)

display(pd.DataFrame([results]).T.rename(columns={0:'value'}))

Processed 80/774
Processed 160/774
Processed 240/774
Processed 320/774
Processed 400/774
Processed 480/774
Processed 560/774
Processed 640/774
Processed 720/774


,value
model,PVTv2-B0 + BiMamba + FPN + ASFF + AdaLoRA + Re...
protocol,cross-dataset zero-shot / no fine-tuning / cla...
external_dataset,PV-Multi-Defect processed
external_split,test
external_split_protocol,fixed 20% finetune / 10% val / 70% test; evalu...
images,774
precision_at_0.25,0.02108
recall_at_0.25,0.017647
f1_at_0.25,0.019212
map50,0.007949


## 12. Generalization drop against the in-domain PVEL-AD TADGNN result

In [17]:
# Try to recover Notebook 12 in-domain test result from its saved output; use the known final report value only if absent.
in_domain_map=None
for csv_path in (TADGNN_BEST_PT.parent.parent).rglob('*.csv'):
    try:
        df=pd.read_csv(csv_path)
        for col in df.columns:
            key=str(col).lower().replace(' ','').replace('_','').replace('-','')
            if 'map5095' in key and len(df)>0:
                vals=pd.to_numeric(df[col],errors='coerce').dropna()
                if len(vals): in_domain_map=float(vals.iloc[-1]); break
        if in_domain_map is not None: break
    except Exception: pass
if in_domain_map is None:
    in_domain_map=0.6065
    source='fallback from saved Notebook 12 final report'
else:
    source='Notebook 12 CSV in Drive'

external_map=results['map50_95']; drop=in_domain_map-external_map
summary=pd.DataFrame([{
    'model':'TADGNN','in_domain_PVEL_AD_mAP50_95':in_domain_map,
    'external_PV_Multi_Defect_mAP50_95':external_map,
    'generalization_drop':drop,'external_fine_tuning':'No','reference_source':source
}])
display(summary)
summary.to_csv(METRICS_DIR/'generalization_drop.csv',index=False)

,model,in_domain_PVEL_AD_mAP50_95,external_PV_Multi_Defect_mAP50_95,generalization_drop,external_fine_tuning,reference_source
0,TADGNN,0.627865,0.005313,0.622552,No,Notebook 12 CSV in Drive


## 13. Save qualitative prediction examples

In [18]:
def draw_example(path,pred,target,out_path):
    im=Image.open(path).convert('RGB'); draw=ImageDraw.Draw(im)
    for b in target['boxes']:
        x1,y1,x2,y2=b.tolist(); draw.rectangle([x1,y1,x2,y2],outline='white',width=3); draw.text((x1,max(0,y1-13)),'GT defect',fill='white')
    keep=pred['scores']>=CONF_THRESHOLD
    for b,s in zip(pred['boxes'][keep],pred['scores'][keep]):
        x1,y1,x2,y2=b.tolist(); draw.rectangle([x1,y1,x2,y2],outline='gray',width=3); draw.text((x1,y1),f'Pred {float(s):.2f}',fill='gray')
    im.save(out_path)

for i,(path,pred,target) in enumerate(all_visual):
    draw_example(path,pred,target,PRED_DIR/f'{i:02d}_{Path(path).name}')
print('Saved examples to:',PRED_DIR)

Saved examples to: /content/drive/MyDrive/Study Materials /CSE465/13_CrossDataset_NoFineTune_TADGNN_70Test/predictions


## 14. Paper-ready zero-shot result table

In [19]:
paper_table=pd.DataFrame([{
    'Model':'PVT-MAAG + TA-DGNN',
    'Source Training':'Modified PVEL-AD',
    'External Test':'PV-Multi-Defect (fixed 70% test)',
    'Fine-tuning':'No',
    'Evaluation':'Class-agnostic defect localization',
    'Precision':round(results['precision_at_0.25'],4),
    'Recall':round(results['recall_at_0.25'],4),
    'F1':round(results['f1_at_0.25'],4),
    'mAP@50':round(results['map50'],4),
    'mAP@50-95':round(results['map50_95'],4),
    'Generalization Drop':round(drop,4),
}])
display(paper_table)
paper_table.to_csv(METRICS_DIR/'paper_ready_cross_dataset_table.csv',index=False)
print('✅ Notebook 13 complete — NO fine-tuning; evaluated ONLY on the fixed 70% external test split.')

,Model,Source Training,External Test,Fine-tuning,Evaluation,Precision,Recall,F1,mAP@50,mAP@50-95,Generalization Drop
0,PVT-MAAG + TA-DGNN,Modified PVEL-AD,PV-Multi-Defect (fixed 70% test),No,Class-agnostic defect localization,0.0211,0.0176,0.0192,0.0079,0.0053,0.6226


✅ Notebook 13 complete — NO fine-tuning; evaluated ONLY on the fixed 70% external test split.


## Interpretation rule

This is the **official no-fine-tuning baseline**. For the later fine-tuned experiment, start from the same PVEL-AD-trained TA-DGNN checkpoint, train only on the fixed 20% `finetune` partition, use the fixed 10% `val` partition for validation/model selection, and evaluate the final model on this exact same untouched 70% `test` partition.